# **1.Preprocessing Section**




In [ ]:
!pip install -q -U keras-tuner


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import tensorflow as tf
import numpy as np
import os
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve
import keras_tuner as kt  # Keras Tuner

In [ ]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

# Define dataset parameters
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your dataset path
img_size = (256, 256)
batch_size = 16

# Custom Layers for DANet Block

In [ ]:

from tensorflow.keras import layers
from tensorflow.keras.utils import register_keras_serializable

@register_keras_serializable()
class PAM(layers.Layer):
    """Position Attention Module"""
    def __init__(self, **kwargs):
        super(PAM, self).__init__(**kwargs)

    def build(self, input_shape):
        self.query_conv = layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.key_conv = layers.Conv2D(input_shape[-1] // 8, kernel_size=1)
        self.value_conv = layers.Conv2D(input_shape[-1], kernel_size=1)
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        query = self.query_conv(inputs)
        key = self.key_conv(inputs)
        value = self.value_conv(inputs)

        query = tf.reshape(query, [tf.shape(query)[0], -1, tf.shape(query)[-1]])
        key = tf.transpose(tf.reshape(key, [tf.shape(key)[0], -1, tf.shape(key)[-1]]), perm=[0, 2, 1])
        energy = tf.matmul(query, key)
        attention = tf.nn.softmax(energy, axis=-1)

        value = tf.reshape(value, [tf.shape(value)[0], -1, tf.shape(value)[-1]])
        out = tf.matmul(attention, value)
        out = tf.reshape(out, tf.shape(inputs))
        return self.gamma * out + inputs

    def get_config(self):
        config = super(PAM, self).get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)



@register_keras_serializable()
class CAM(layers.Layer):
    """Channel Attention Module"""
    def __init__(self, **kwargs):
        super(CAM, self).__init__(**kwargs)

    def build(self, input_shape):
        self.gamma = self.add_weight(name="gamma", shape=(), initializer="zeros", trainable=True)

    def call(self, inputs):
        query = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])
        key = tf.transpose(query, perm=[0, 2, 1])
        energy = tf.matmul(key, query)
        attention = tf.nn.softmax(energy, axis=-1)

        value = tf.reshape(inputs, [tf.shape(inputs)[0], -1, tf.shape(inputs)[-1]])
        out = tf.matmul(value, attention)
        out = tf.reshape(out, tf.shape(inputs))
        return self.gamma * out + inputs

    def get_config(self):
        config = super(CAM, self).get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)


In [ ]:
@register_keras_serializable()
class DANetBlock(layers.Layer):
    """Dual Attention Network Block"""
    def __init__(self, **kwargs):
        super(DANetBlock, self).__init__(**kwargs)
        self.pam = PAM()
        self.cam = CAM()

    def call(self, inputs):
        pam_out = self.pam(inputs)
        cam_out = self.cam(inputs)
        return pam_out + cam_out

    def get_config(self):
        config = super(DANetBlock, self).get_config()
        return config

    @classmethod
    def from_config(cls, config):
        return cls(**config)

# -------------------------------
# Data Augmentation & Normalization
# -------------------------------

In [ ]:

data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical"),
    tf.keras.layers.RandomRotation(0.2),
    tf.keras.layers.RandomZoom(0.2),
    tf.keras.layers.RandomContrast(0.2),
    tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
    tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
])

normalization_layer = tf.keras.layers.Rescaling(1./255)

# -------------------------------
# Prepare Datasets with Custom Splits
# -------------------------------

In [ ]:

def prepare_datasets(batch_size):
    # First, split the data into 80% (train+validation) and 20% test using image_dataset_from_directory
    train_val_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    test_ds = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Now, split the 80% train_val_ds into training (90%) and validation (10%) sets
    total_batches = tf.data.experimental.cardinality(train_val_ds).numpy()
    num_val_batches = max(1, int(0.1 * total_batches))
    val_ds = train_val_ds.take(num_val_batches)
    train_ds = train_val_ds.skip(num_val_batches)

    # Apply augmentation and normalization:
    train_ds = train_ds.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
    test_ds = test_ds.map(lambda x, y: (normalization_layer(x), y))

    return train_ds, val_ds, test_ds

# Create the datasets
train_ds, val_ds, test_ds = prepare_datasets(batch_size=batch_size)


# **2.Training Section**

In [ ]:


# Build the CNN model with the DANet block
def build_model():
    inputs = tf.keras.Input(shape=(256, 256, 3))

    # Convolutional Block 1
    x = tf.keras.layers.Conv2D(64, (3, 3), activation='relu', padding='same')(inputs)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 2
    x = tf.keras.layers.Conv2D(128, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # Convolutional Block 3
    x = tf.keras.layers.Conv2D(256, (3, 3), activation='relu', padding='same')(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.MaxPooling2D((2, 2))(x)

    # DANet Block
    x = DANetBlock()(x)

    # Global Average Pooling and Fully Connected Layers
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.Dense(256, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)
    x = tf.keras.layers.Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01))(x)
    x = tf.keras.layers.Dropout(0.4)(x)

    # Output Layer
    outputs = tf.keras.layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=outputs)
    learning_rate = 0.0004
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [ ]:
# Optionally, test the DANetBlock with a random input to verify its output shape
x_temp = tf.random.normal((1, 64, 64, 256))
danet_block = DANetBlock()
x_out = danet_block(x_temp)
print(f"DANetBlock output shape: {x_out.shape}")


In [ ]:
# Define Early Stopping callback
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
# Build and train the model
model = build_model()
history = model.fit(train_ds, validation_data=val_ds, epochs=60, callbacks=[early_stopping])

In [ ]:
# Plot training accuracy and loss
plt.figure(figsize=(12, 6))
plt.plot(history.history['accuracy'], label='Training Accuracy')
plt.plot(history.history['val_accuracy'], label='Validation Accuracy')
plt.title('Model Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(12, 6))
plt.plot(history.history['loss'], label='Training Loss')
plt.plot(history.history['val_loss'], label='Validation Loss')
plt.title('Model Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.legend()
plt.grid()
plt.show()


# **3.Testing Section**

In [ ]:


# Evaluate the model on the test dataset
y_true = []
y_pred_probs = []

for batch in test_ds.as_numpy_iterator():
    X, y = batch
    preds = model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(preds)

y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs).flatten()
y_pred = (y_pred_probs > 0.5).astype(int)

# Compute evaluation metrics
accuracy = accuracy_score(y_true, y_pred)
precision = precision_score(y_true, y_pred)
recall = recall_score(y_true, y_pred)
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_pred_probs)

print(f"Test Accuracy: {accuracy:.4f}")
print(f"Test Precision: {precision:.4f}")
print(f"Test Recall: {recall:.4f}")
print(f"Test F1 Score: {f1:.4f}")
print(f"Test AUC-ROC: {roc_auc:.4f}")


In [ ]:
# Plot Confusion Matrix
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Class 0", "Class 1"])
disp.plot(cmap='Blues')
plt.title('Confusion Matrix on Test Set')
plt.show()

In [ ]:
# Plot ROC Curve
fpr, tpr, _ = roc_curve(y_true, y_pred_probs)
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, label=f'ROC Curve (AUC = {roc_auc:.4f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve on Test Set')
plt.legend(loc="lower right")
plt.grid()
plt.show()

In [ ]:
# Plot Precision-Recall Curve
from sklearn.metrics import precision_recall_curve
precision_vals, recall_vals, _ = precision_recall_curve(y_true, y_pred_probs)
pr_auc = np.trapz(precision_vals, recall_vals)
plt.figure(figsize=(10, 6))
plt.plot(recall_vals, precision_vals, label=f'PR Curve (AUC = {pr_auc:.4f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve on Test Set')
plt.legend(loc="upper right")
plt.grid()
plt.show()

*save the model*

In [ ]:

directory_path = '/content/drive/My Drive/saved_model'
model_path = os.path.join(directory_path, 'cnn_model_breast_github.keras')
os.makedirs(directory_path, exist_ok=True)
model.save(model_path)
print(f"Model saved at: {model_path}")